# Speaker Verification Pipeline: ECAPA-TDNN with ArcFace Metric Learning

This notebook implements an end-to-end voice biometrics and speaker verification pipeline using ECAPA-TDNN (Emphasized Channel Attention, Propagation and Aggregation Time-Delay Neural Network) with Additive Angular Margin (ArcFace) loss on the LibriSpeech dataset (`torchaudio.datasets.LIBRISPEECH`).

The code is organized into sequential functional stages:
1. Environment setup and dependency installation
2. Telemetry dashboard extension
3. Core library imports and environment setup
4. Compute device allocation
5. Hyperparameter and directory configuration
6. Multi-speaker dataset ingestion and preprocessing
7. Acoustic feature extraction with CMVN and SpecAugment
8. Exploratory multi-speaker acoustic inspection
9. Neural network architecture definition
10. Additive Angular Margin (ArcFace) loss formulation
11. Model instantiation and parameter summary
12. Training and validation loop execution
13. Speaker verification evaluation (Equal Error Rate and DET curve)
14. Speaker latent space visualization and verification demo

## 1. Environment Setup and Dependency Installation

Install all required runtime dependencies including core deep learning libraries, accelerator backends, telemetry tooling, and metric evaluation libraries.

In [ ]:
%pip install -q torch torchvision torchaudio torch_xla tensorboard matplotlib pandas ipywidgets scipy scikit-learn seaborn safetensors

## 2. Telemetry Dashboard Extension

Load the inline telemetry viewer to inspect training progression, ArcFace loss, and EER metrics directly within the execution environment.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports and Environment Setup

Import numerical processing, metric learning modules, audio transforms, accelerator runtime interfaces, and visualization tools. Set deterministic random seeds.

In [ ]:
import json
import os
import math
import random
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchaudio
from torchaudio.datasets import LIBRISPEECH
import torchaudio.transforms as T
import IPython.display as ipd
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(f"PyTorch: {torch.__version__} | TorchAudio: {torchaudio.__version__}")


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)
        if loss_fn is not None and len(loss_fn.state_dict()) > 0:
            loss_dict = {
                k: v.detach().cpu().contiguous()
                for k, v in loss_fn.state_dict().items()
            }
            zf.writestr("loss_fn.safetensors", safetensors_save(loss_dict))


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
    loss_fn: torch.nn.Module | None = None,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")
        if loss_fn is not None and "loss_fn.safetensors" in zf.namelist():
            loss_bytes = zf.read("loss_fn.safetensors")
            loss_dict = safetensors_load(loss_bytes)
            loss_dict = {k: v.to(device) for k, v in loss_dict.items()}
            loss_fn.load_state_dict(loss_dict)

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)

    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Initialize and allocate the compute accelerator (Cloud TPU via PyTorch XLA, NVIDIA CUDA GPU, Apple Silicon MPS, or CPU).

In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"

print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Define file system locations for data caching, telemetry logs, and checkpoints. Set acoustic parameters ($16\text{ kHz}$, $80$ Mel bins), ECAPA channel dimensions ($C=512$), embedding size ($192$), and ArcFace hyperparameters ($s=30, m=0.20$).

In [ ]:
# ── Directory Paths ────────────────────────────────────────────────────────
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/audio_speaker_verification_ecapa").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/audio").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "speaker_ecapa_tdnn.zip"

# ── Signal & Feature Hyperparameters ────────────────────────────────────
SAMPLE_RATE = 16000
SEGMENT_DURATION = 2.5
NUM_SAMPLES = int(SAMPLE_RATE * SEGMENT_DURATION)
N_FFT = 512
HOP_LENGTH = 160
N_MELS = 80

# ── ECAPA-TDNN & ArcFace Hyperparameters ────────────────────────────────
CHANNELS = 512
EMBED_DIM = 192
SCALE = 30.0
MARGIN = 0.20

# ── Optimization Parameters ─────────────────────────────────────────────
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 2e-5
NUM_WORKERS = 2 if os.name != "nt" else 0
PIN_MEMORY = device.type == "cuda"

print(
    f"ECAPA Config: Channels={CHANNELS}, Embed_Dim={EMBED_DIM}, ArcFace (s={SCALE}, m={MARGIN})"
)

## 6. Multi-Speaker Dataset Ingestion and Preprocessing

Extract and index individual speaker utterances across the LibriSpeech corpus, establishing speaker identifier mappings and building a multi-speaker dataset with random fixed-duration temporal cropping ($2.5\text{ s}$).

In [ ]:
class SpeakerVerificationDataset(Dataset):
    """Dataset that samples fixed-duration audio slices indexed by speaker ID."""

    def __init__(
        self, root_dir, download=True, segment_samples=40000, is_training=True
    ):
        self.segment_samples = segment_samples
        self.is_training = is_training
        self.librispeech = LIBRISPEECH(
            root=str(root_dir), url="dev-clean", download=download
        )

        # Group utterance indices by speaker ID
        self.speaker_to_indices = {}
        for idx in range(len(self.librispeech)):
            _, _, _, spk_id, _, _ = self.librispeech[idx]
            if spk_id not in self.speaker_to_indices:
                self.speaker_to_indices[spk_id] = []
            self.speaker_to_indices[spk_id].append(idx)

        self.speakers = sorted(list(self.speaker_to_indices.keys()))
        self.spk_to_idx = {spk: i for i, spk in enumerate(self.speakers)}
        self.num_speakers = len(self.speakers)
        print(
            f"Indexed {len(self.librispeech):,} utterances across {self.num_speakers} unique speakers."
        )

    def __len__(self):
        return len(self.librispeech)

    def __getitem__(self, idx):
        waveform, sr, _, spk_id, _, _ = self.librispeech[idx]
        if sr != SAMPLE_RATE:
            resampler = T.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
            waveform = resampler(waveform)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        length = waveform.shape[1]
        if length < self.segment_samples:
            waveform = F.pad(
                waveform, (0, self.segment_samples - length), mode="constant", value=0.0
            )
        elif length > self.segment_samples:
            start = (
                random.randint(0, length - self.segment_samples)
                if self.is_training
                else 0
            )
            waveform = waveform[:, start : start + self.segment_samples]

        return waveform, self.spk_to_idx[spk_id], spk_id


# Instantiate dataset
full_dataset = SpeakerVerificationDataset(
    DATA_DIR, download=True, segment_samples=NUM_SAMPLES, is_training=True
)

# 80/20 train/validation split
n_train = int(0.80 * len(full_dataset))
n_val = len(full_dataset) - n_train
train_set, val_set = torch.utils.data.random_split(
    full_dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

## 7. Acoustic Feature Extraction with CMVN and SpecAugment

Extract 80-channel log Mel-filterbank spectrograms ($N_{\text{fft}}=512, \text{hop}=160$) with Cepstral Mean and Variance Normalization (CMVN) and SpecAugment time/frequency masking.

In [ ]:
class SpeakerFeatureExtractor(nn.Module):
    """Extracts Log-Mel Filterbanks with CMVN and SpecAugment."""

    def __init__(
        self, sample_rate=16000, n_fft=512, hop_length=160, n_mels=80, is_training=False
    ):
        super().__init__()
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            win_length=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            power=2.0,
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80.0)
        self.is_training = is_training
        self.freq_mask = T.FrequencyMasking(freq_mask_param=12)
        self.time_mask = T.TimeMasking(time_mask_param=25)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, 1, Samples]
        mel = self.mel_transform(x)
        log_mel = self.amplitude_to_db(mel).squeeze(1)

        # Cepstral Mean & Variance Normalization (CMVN) across frames
        mean = log_mel.mean(dim=-1, keepdim=True)
        std = log_mel.std(dim=-1, keepdim=True) + 1e-6
        log_mel = (log_mel - mean) / std

        if self.is_training and self.training:
            log_mel = self.freq_mask(log_mel.unsqueeze(1)).squeeze(1)
            log_mel = self.time_mask(log_mel.unsqueeze(1)).squeeze(1)

        return log_mel

## 8. Exploratory Multi-Speaker Acoustic Inspection

Inspect speech waveforms, spectrograms, and pitch dynamics across distinct speaker identities with interactive playback widgets.

In [ ]:
sample_waves, sample_labels, sample_spk_ids = next(iter(train_loader))
feature_extractor = SpeakerFeatureExtractor(is_training=False)
sample_mels = feature_extractor(sample_waves)

fig, axes = plt.subplots(3, 2, figsize=(14, 8))
time_axis = np.linspace(0, SEGMENT_DURATION, NUM_SAMPLES)

for i in range(3):
    spk_id = sample_spk_ids[i].item()
    # Waveform
    axes[i, 0].plot(time_axis, sample_waves[i, 0].numpy(), color="#2ca02c", lw=0.8)
    axes[i, 0].set_title(f"Speaker ID: {spk_id} - Waveform")
    axes[i, 0].set_ylabel("Amplitude")
    axes[i, 0].grid(True, alpha=0.3)

    # Mel Spectrogram
    im = axes[i, 1].imshow(
        sample_mels[i].numpy(), origin="lower", aspect="auto", cmap="viridis"
    )
    axes[i, 1].set_title(f"Normalized Mel-Spectrogram (CMVN)")
    axes[i, 1].set_ylabel("Mel Bin")
    fig.colorbar(im, ax=axes[i, 1])

axes[2, 0].set_xlabel("Time (s)")
axes[2, 1].set_xlabel("Time Frames (10ms)")
plt.tight_layout()
plt.show()

# Interactive Audio Playback
print(f"Listen to Speaker {sample_spk_ids[0].item()}:")
ipd.Audio(sample_waves[0, 0].numpy(), rate=SAMPLE_RATE)

## 9. Neural Network Architecture Definition

Construct the ECAPA-TDNN speaker embedding architecture:
- **Stem TDNN**: 1D convolution ($k=5, d=1, C=512$) with BatchNorm and ReLU.
- **Squeeze-and-Excitation Res2Net Blocks**: Multi-scale dilated 1D convolutions ($k=3, d \in \{2, 3, 4\}$) with 1D Squeeze-and-Excitation channel attention.
- **Multi-Layer Feature Aggregation (MFA)**: Concatenates intermediate feature maps ($3 \times 512 = 1536$).
- **Attentive Statistics Pooling (ASP)**: Computes channel-wise attentive weighted mean $\boldsymbol{\mu}$ and weighted standard deviation $\boldsymbol{\sigma}$ $\to 3072$-d pooled representation.
- **Embedding Head**: Fully connected layer with BatchNorm producing $192$-d $L_2$-normalized speaker embeddings.

In [ ]:
class SqueezeExcitationTDNN(nn.Module):
    """Squeeze-and-Excitation Block for 1D Frame Sequences."""

    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        self.fc1 = nn.Conv1d(channels, channels // reduction, kernel_size=1)
        self.relu = nn.ReLU()
        self.fc2 = nn.Conv1d(channels // reduction, channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T]
        w = x.mean(dim=-1, keepdim=True)
        w = self.sigmoid(self.fc2(self.relu(self.fc1(w))))
        return x * w


class Res2NetBlock(nn.Module):
    """Res2Net Block with Dilated 1D Convolutions and SE Attention."""

    def __init__(self, channels: int, scale: int = 4, dilation: int = 1):
        super().__init__()
        self.scale = scale
        self.width = channels // scale

        self.conv1 = nn.Conv1d(channels, channels, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()

        self.convs = nn.ModuleList(
            [
                nn.Conv1d(
                    self.width,
                    self.width,
                    kernel_size=3,
                    padding=dilation,
                    dilation=dilation,
                )
                for _ in range(scale - 1)
            ]
        )
        self.bns = nn.ModuleList([nn.BatchNorm1d(self.width) for _ in range(scale - 1)])

        self.conv3 = nn.Conv1d(channels, channels, kernel_size=1)
        self.bn3 = nn.BatchNorm1d(channels)
        self.se = SqueezeExcitationTDNN(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))

        # Split into sub-feature maps
        xs = torch.split(out, self.width, dim=1)
        ys = [xs[0]]
        for i in range(self.scale - 1):
            if i == 0:
                y = self.relu(self.bns[i](self.convs[i](xs[i + 1])))
            else:
                y = self.relu(self.bns[i](self.convs[i](xs[i + 1] + ys[i])))
            ys.append(y)

        out = torch.cat(ys, dim=1)
        out = self.bn3(self.conv3(out))
        out = self.se(out)
        return self.relu(out + residual)


class AttentiveStatisticsPooling(nn.Module):
    """Attentive Statistics Pooling (ASP) for frame-level aggregation."""

    def __init__(self, channels: int, attention_channels: int = 128):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv1d(channels, attention_channels, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm1d(attention_channels),
            nn.Conv1d(attention_channels, channels, kernel_size=1),
            nn.Softmax(dim=-1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T]
        alpha = self.attn(x)
        mean = torch.sum(alpha * x, dim=-1)

        # Weighted Variance and Standard Deviation
        diff = x - mean.unsqueeze(-1)
        var = torch.sum(alpha * (diff**2), dim=-1)
        std = torch.sqrt(var.clamp(min=1e-8))

        # Concatenate mean and standard deviation: [B, 2*C]
        return torch.cat([mean, std], dim=1)


class ECAPATDNN(nn.Module):
    """ECAPA-TDNN Speaker Embedding Extractor."""

    def __init__(
        self, in_channels: int = 80, channels: int = 512, embed_dim: int = 192
    ):
        super().__init__()
        self.feature_extractor = SpeakerFeatureExtractor(is_training=True)

        # Initial TDNN layer
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
        )

        # 3 Res2Net Blocks with varying dilations (2, 3, 4)
        self.block1 = Res2NetBlock(channels, dilation=2)
        self.block2 = Res2NetBlock(channels, dilation=3)
        self.block3 = Res2NetBlock(channels, dilation=4)

        # Multi-layer Feature Aggregation (MFA)
        self.mfa = nn.Sequential(
            nn.Conv1d(3 * channels, 3 * channels, kernel_size=1),
            nn.BatchNorm1d(3 * channels),
            nn.ReLU(),
        )

        # Attentive Statistics Pooling
        self.asp = AttentiveStatisticsPooling(3 * channels)
        self.asp_bn = nn.BatchNorm1d(6 * channels)

        # Linear Embedding Projection
        self.fc = nn.Linear(6 * channels, embed_dim)
        self.bn_out = nn.BatchNorm1d(embed_dim)

    def forward(self, waveforms: torch.Tensor) -> torch.Tensor:
        # waveforms: [B, 1, Samples]
        mels = self.feature_extractor(waveforms)
        x = self.stem(mels)

        o1 = self.block1(x)
        o2 = self.block2(o1)
        o3 = self.block3(o2)

        # Multi-layer concatenation
        mfa_in = torch.cat([o1, o2, o3], dim=1)
        mfa_out = self.mfa(mfa_in)

        # Pooling & projection
        stats = self.asp_bn(self.asp(mfa_out))
        embedding = self.bn_out(self.fc(stats))

        # L2 Unit Normalization
        embedding_norm = F.normalize(embedding, p=2, dim=1)
        return embedding_norm

## 10. Additive Angular Margin (ArcFace) Loss Formulation

Formulate Additive Angular Margin (ArcFace) loss for deep metric learning:
$$\mathcal{L}_{\text{ArcFace}} = -\ln \frac{e^{s \cdot \cos(\theta_y + m)}}{e^{s \cdot \cos(\theta_y + m)} + \sum_{j \neq y} e^{s \cdot \cos\theta_j}}$$

In [ ]:
class ArcFaceLoss(nn.Module):
    """Additive Angular Margin Loss (ArcFace)."""

    def __init__(
        self, in_features: int, num_classes: int, s: float = 30.0, m: float = 0.20
    ):
        super().__init__()
        self.in_features = in_features
        self.num_classes = num_classes
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        # embeddings: [B, D] (already L2 normalized)
        # Normalize class weight vectors
        norm_weights = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(embeddings, norm_weights).clamp(-1.0 + 1e-7, 1.0 - 1e-7)

        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(0, 1)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s

        return F.cross_entropy(output, labels)

## 11. Model Instantiation and Parameter Summary

Instantiate the ECAPA-TDNN network on the compute device, audit total trainable parameters, and verify $L_2$ unit-norm embedding output.

In [ ]:
ecapa_model = ECAPATDNN(in_channels=N_MELS, channels=CHANNELS, embed_dim=EMBED_DIM).to(
    device
)
arcface_loss_fn = ArcFaceLoss(
    in_features=EMBED_DIM, num_classes=full_dataset.num_speakers, s=SCALE, m=MARGIN
).to(device)

total_params = sum(p.numel() for p in ecapa_model.parameters()) + sum(
    p.numel() for p in arcface_loss_fn.parameters()
)
print(f"ECAPA-TDNN Architecture Summary:")
print(f"Total Parameters: {total_params:,} ({total_params * 4 / (1024**2):.2f} MB)")

# Verify forward pass
dummy_wave = torch.randn(4, 1, NUM_SAMPLES, device=device)
dummy_labels = torch.tensor([0, 1, 2, 3], device=device)

with torch.no_grad():
    dummy_emb = ecapa_model(dummy_wave)
    dummy_loss = arcface_loss_fn(dummy_emb, dummy_labels)

norms = torch.norm(dummy_emb, p=2, dim=1)
print(f"Embedding Shape: {dummy_emb.shape} | L2 Norms: {norms.cpu().numpy()}")
print(f"Dummy ArcFace Loss: {dummy_loss.item():.4f}")
assert torch.allclose(norms, torch.ones_like(norms)), (
    "Embeddings are not unit normalized!"
)

## 12. Training and Validation Loop Execution

Execute the optimization loop with `AdamW` optimizer, StepLR decay, gradient norm clipping ($5.0$), ArcFace validation loss tracking, and checkpoint persistence.

In [ ]:
optimizer = torch.optim.AdamW(
    list(ecapa_model.parameters()) + list(arcface_loss_fn.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

writer = SummaryWriter(log_dir=str(LOG_DIR))
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None


def evaluate(model, loss_fn, dataloader, device):
    model.eval()
    loss_fn.eval()
    total_loss, total_samples = 0.0, 0
    with torch.no_grad():
        for waveforms, labels, _ in dataloader:
            waveforms, labels = waveforms.to(device), labels.to(device)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                embeddings = model(waveforms)
                loss = loss_fn(embeddings, labels)
            total_loss += loss.item() * waveforms.size(0)
            total_samples += waveforms.size(0)
    return total_loss / total_samples


best_val_loss = float("inf")
print(f"Starting ECAPA-TDNN Training for {EPOCHS} epochs on {device_name}...")

for epoch in range(1, EPOCHS + 1):
    ecapa_model.train()
    arcface_loss_fn.train()
    train_loss, train_samples = 0.0, 0

    for waveforms, labels, _ in train_loader:
        waveforms, labels = waveforms.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast("cuda"):
                embeddings = ecapa_model(waveforms)
                loss = arcface_loss_fn(embeddings, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(ecapa_model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            embeddings = ecapa_model(waveforms)
            loss = arcface_loss_fn(embeddings, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ecapa_model.parameters(), max_norm=5.0)
            if is_xla:
                xm.optimizer_step(optimizer)
            else:
                optimizer.step()

        train_loss += loss.item() * waveforms.size(0)
        train_samples += waveforms.size(0)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    epoch_train_loss = train_loss / train_samples
    epoch_val_loss = evaluate(ecapa_model, arcface_loss_fn, val_loader, device)

    writer.add_scalar("Loss/Train_ArcFace", epoch_train_loss, epoch)
    writer.add_scalar("Loss/Val_ArcFace", epoch_val_loss, epoch)
    writer.flush()

    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"LR: {current_lr:.2e}"
    )

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        save_model_zip(
            model=ecapa_model,
            metadata={
                "epoch": epoch,
                "val_loss": best_val_loss,
            },
            save_path=MODEL_PATH,
            loss_fn=arcface_loss_fn,
            is_xla=is_xla,
        )
        print(
            f"  --> Saved new best checkpoint to {MODEL_PATH} (Val Loss: {best_val_loss:.4f})"
        )

writer.close()
print("Training completed.")

## 13. Speaker Verification Evaluation (Equal Error Rate and DET Curve)

Evaluate speaker verification performance on genuine (same-speaker) and imposter (cross-speaker) trial pairs, computing cosine similarity distributions, Equal Error Rate (EER), and Detection Error Tradeoff (DET) curves.

In [ ]:
if MODEL_PATH.exists():
    checkpoint_meta = load_model_zip(
        ecapa_model, MODEL_PATH, device=device, loss_fn=arcface_loss_fn
    )
    print(
        f"Loaded best checkpoint from Epoch {checkpoint_meta['epoch']} (Val Loss: {checkpoint_meta['val_loss']:.4f})"
    )


def extract_embeddings(model, dataset):
    model.eval()
    embeddings, speaker_ids = [], []
    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=NUM_WORKERS,
        persistent_workers=(NUM_WORKERS > 0),
    )

    with torch.no_grad():
        for waves, _, spk_ids in loader:
            waves = waves.to(device)
            emb = model(waves).cpu().numpy()
            embeddings.extend(emb)
            speaker_ids.extend(spk_ids.numpy())

    return np.array(embeddings), np.array(speaker_ids)


val_embs, val_spk_ids = extract_embeddings(ecapa_model, val_set)


# Generate genuine and imposter verification trials
def generate_verification_trials(embeddings, labels, n_trials=5000):
    unique_speakers = np.unique(labels)
    spk_to_idx = {s: np.where(labels == s)[0] for s in unique_speakers}

    scores, ground_truth = [], []

    for _ in range(n_trials // 2):
        # Genuine pair (same speaker)
        spk = random.choice([s for s in unique_speakers if len(spk_to_idx[s]) >= 2])
        idx1, idx2 = np.random.choice(spk_to_idx[spk], 2, replace=False)
        sim = np.dot(embeddings[idx1], embeddings[idx2])
        scores.append(sim)
        ground_truth.append(1)

        # Imposter pair (different speakers)
        spk1, spk2 = np.random.choice(unique_speakers, 2, replace=False)
        idx1 = random.choice(spk_to_idx[spk1])
        idx2 = random.choice(spk_to_idx[spk2])
        sim = np.dot(embeddings[idx1], embeddings[idx2])
        scores.append(sim)
        ground_truth.append(0)

    return np.array(scores), np.array(ground_truth)


from sklearn.metrics import roc_curve

scores, targets = generate_verification_trials(val_embs, val_spk_ids, n_trials=4000)
fpr, tpr, thresholds = roc_curve(targets, scores)
fnr = 1 - tpr

# Find Equal Error Rate (EER) where FPR == FNR
eer_idx = np.nanargmin(np.absolute((fnr - fpr)))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0 * 100.0
eer_threshold = thresholds[eer_idx]

print(f"\nSpeaker Verification Performance:")
print(f"Equal Error Rate (EER): {eer:.2f}% (at Cosine Threshold = {eer_threshold:.4f})")

# Plot ROC & Score Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# Score Distribution
ax1.hist(
    scores[targets == 1],
    bins=40,
    alpha=0.6,
    label="Genuine (Same Speaker)",
    color="green",
)
ax1.hist(
    scores[targets == 0],
    bins=40,
    alpha=0.6,
    label="Imposter (Diff Speaker)",
    color="red",
)
ax1.axvline(
    eer_threshold,
    color="black",
    linestyle="--",
    label=f"EER Threshold ({eer_threshold:.2f})",
)
ax1.set_title("Cosine Similarity Score Distribution")
ax1.set_xlabel("Cosine Similarity Score")
ax1.set_ylabel("Count")
ax1.legend()
ax1.grid(True, alpha=0.3)

# ROC Curve
ax2.plot(fpr, tpr, color="blue", lw=2, label=f"ROC Curve (EER = {eer:.2f}%)")
ax2.plot([0, 1], [0, 1], color="gray", linestyle="--")
ax2.scatter(
    fpr[eer_idx],
    tpr[eer_idx],
    color="red",
    s=80,
    zorder=5,
    label=f"EER Point ({eer:.2f}%)",
)
ax2.set_title("Speaker Verification ROC Curve")
ax2.set_xlabel("False Acceptance Rate (FAR / FPR)")
ax2.set_ylabel("True Acceptance Rate (TAR / TPR)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Speaker Latent Space Visualization and Verification Demo

Visualize 2D t-SNE projections of learned 192-dimensional speaker embeddings demonstrating identity clustering, and provide an interactive verification authentication demo.

In [ ]:
from sklearn.manifold import TSNE

# Select top 8 speakers with most utterances
top_speakers = pd.Series(val_spk_ids).value_counts().head(8).index.tolist()
mask = np.isin(val_spk_ids, top_speakers)

tsne = TSNE(n_components=2, perplexity=15, random_state=42)
embs_2d = tsne.fit_transform(val_embs[mask])
labels_subset = val_spk_ids[mask]

plt.figure(figsize=(10, 8))
for spk in top_speakers:
    spk_mask = labels_subset == spk
    plt.scatter(
        embs_2d[spk_mask, 0],
        embs_2d[spk_mask, 1],
        label=f"Speaker {spk}",
        s=50,
        alpha=0.85,
    )

plt.title("2D t-SNE Projection of 192-d ECAPA-TDNN Speaker Embeddings")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


def verify_speaker_pair(
    audio_1: torch.Tensor, audio_2: torch.Tensor, threshold: float = eer_threshold
):
    """Authenticates whether two audio waveforms belong to the same speaker."""
    ecapa_model.eval()
    w1 = audio_1.unsqueeze(0).to(device)
    w2 = audio_2.unsqueeze(0).to(device)

    with torch.no_grad():
        e1 = ecapa_model(w1).cpu().numpy()[0]
        e2 = ecapa_model(w2).cpu().numpy()[0]

    similarity = np.dot(e1, e2)
    is_match = similarity >= threshold

    print("=" * 60)
    print(f"Cosine Similarity Score: {similarity:.4f} (EER Threshold: {threshold:.4f})")
    print(
        f"Biometric Decision: {'>>> SAME SPEAKER (MATCH) <<<' if is_match else '>>> DIFFERENT SPEAKER (IMPOSTER) <<<'}"
    )
    print("=" * 60)
    return is_match


# Demo on a genuine pair
sample_spk = top_speakers[0]
spk_indices = np.where(val_spk_ids == sample_spk)[0]
wave_ref, _, _ = val_set[spk_indices[0]]
wave_query, _, _ = val_set[spk_indices[1]]

print(f"Testing Genuine Match for Speaker {sample_spk}:")
verify_speaker_pair(wave_ref, wave_query)
display(ipd.Audio(wave_ref[0].numpy(), rate=SAMPLE_RATE))